# Problema X23B10T0 — Solução via Funções de Green

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x23b10t0.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Placa plana de espessura $L$, com:

- **Contorno $x = 0$** (tipo 2): fluxo de calor prescrito $q''_0$
- **Contorno $x = L$** (tipo 3): convecção com coeficiente $h_2$ e temperatura ambiente $T_\infty = 0$
- **Condição inicial:** $T(x,0) = 0$

A notação X23 indica: geometria plana (X), fluxo prescrito em $x=0$ (2) e convecção em $x=L$ (3).

## Solução analítica

Como $T(x,0) = 0$, a solução em termos de funções de Green é:

$$T(x,t) = \frac{q''_0(L-x)}{k} + \frac{q''_0}{h_2}
- \frac{\alpha q''_0}{k}\sum_{m=1}^{\infty}
C_m\,\frac{\cos(\beta_m x/L)}{A_m}\,\bigl(1 - e^{-A_m t}\bigr)$$

onde os autovalores $\beta_m$ são raízes da equação transcendental:

$$\beta_m\,\tan(\beta_m) = H_2, \qquad H_2 = \frac{h_2 L}{k}$$

e os coeficientes de normalização são:

$$A_m = \frac{\alpha\,\beta_m^2}{L^2}, \qquad
C_m = \frac{2(\beta_m^2 + H_2^2)}{L(\beta_m^2 + H_2^2 + H_2)}$$

A parcela $q''_0(L-x)/k + q''_0/h_2$ é o regime permanente: o perfil linear de temperatura que seria atingido assintoticamente para $t \to \infty$.

## Bibliotecas utilizadas

Para avaliar a solução analítica computacionalmente, utilizamos três bibliotecas Python:

- **NumPy** (`numpy`): biblioteca para computação numérica. Fornece vetores e matrizes eficientes e funções matemáticas como `np.sqrt`, `np.exp`, `np.cos`, `np.linspace`.
- **Matplotlib** (`matplotlib.pyplot`): biblioteca para criação de gráficos 2D.
- **SciPy** (`scipy.optimize.brentq`): fornece o método de Brent para encontrar raízes de funções em um intervalo. É usado aqui para calcular numericamente os autovalores transcendentais $\beta_m$.

In [ ]:
import numpy as np                        # computação numérica
import matplotlib.pyplot as plt           # gráficos
from scipy.optimize import brentq         # raízes de funções (método de Brent)

## Parâmetros do problema

Os parâmetros físicos são armazenados como **variáveis** Python. Uma variável é criada com o sinal `=`, por exemplo `L = 66e-4` define $L = 0{,}0066$ m (a notação `66e-4` é equivalente a $66 \times 10^{-4}$). Os comentários após `#` não são executados e servem apenas para documentar o código.

O parâmetro adimensional $H_2 = h_2 L / k$ resume o balanço entre a condução interna e a convecção na superfície $x = L$. Quanto maior $H_2$, mais eficiente é a dissipação convectiva.

In [ ]:
L     = 66e-4        # espessura da placa [m]  (66e-4 = 0,0066 m)
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e3          # fluxo de calor imposto em x=0 [W/m²]
h2    = 25.0         # coeficiente convectivo em x=L [W/(m²·K)]

H2    = h2 * L / k  # parâmetro adimensional de Biot
M     = 300          # número de termos da série (truncamento)

print(f'H2 = h2·L/k = {H2:.4f}')

## Autovalores transcendentais e o método de Brent

Diferentemente dos problemas X21 e X22 — cujos autovalores têm fórmulas explícitas ($\beta_m = (m-\tfrac{1}{2})\pi$ e $\beta_m = m\pi$, respectivamente) —, no Problema X23 os autovalores satisfazem a equação transcendental:

$$\beta_m \tan(\beta_m) = H_2$$

Essa equação não tem solução em forma fechada: cada raiz $\beta_m$ precisa ser encontrada **numericamente**. Para isso utilizamos a função `brentq` da SciPy.

### O que `brentq` faz

`brentq(f, a, b)` encontra um zero de uma função `f` no intervalo `(a, b)`, ou seja, encontra o valor de $x$ tal que $f(x) = 0$. A função garante encontrar a raiz **se** $f(a)$ e $f(b)$ tiverem sinais opostos — o que indica que a função troca de sinal dentro do intervalo e portanto há uma raiz lá.

### O método de Brent

O método combina três estratégias para ser ao mesmo tempo robusto e rápido:

1. **Bisseção** — divide o intervalo ao meio; lenta, mas nunca falha.
2. **Interpolação linear** (secante) — usa a reta entre dois pontos para estimar a raiz; mais rápida.
3. **Interpolação quadrática inversa** — usa três pontos para uma estimativa ainda mais precisa.

A cada passo ele tenta a estratégia mais rápida; se o resultado não for confiável, recua para a bisseção. Por isso é considerado o melhor algoritmo de propósito geral para raízes em um intervalo.

### Por que não Newton-Raphson?

Newton-Raphson é mais rápido quando converge, mas pode divergir se o chute inicial for ruim ou se a derivada for quase zero. Como aqui há singularidades da tangente espalhadas pelo domínio, `brentq` é a escolha mais segura — a garantia de convergência é mais importante do que a velocidade.

### Busca por intervalos

A tangente tem singularidades em $\beta = \pi/2,\; 3\pi/2,\; 5\pi/2, \ldots$ — por isso cada raiz é buscada no seu próprio ramo:

| $m$ | Intervalo de busca |
|---|---|
| 1 | $(0,\ \pi/2)$ |
| 2 | $(\pi,\ 3\pi/2)$ |
| 3 | $(2\pi,\ 5\pi/2)$ |

O parâmetro `eps = 1e-10` evita avaliar `tan` exatamente nas singularidades, onde ela vai a $\pm\infty$.

Por fim:
- `np.array([...])` converte a lista de raízes em um vetor NumPy, permitindo que operações matemáticas sejam aplicadas a todos os elementos de uma vez.
- `np.linspace(inicio, fim, N)` cria um vetor com `N` valores reais igualmente espaçados entre `inicio` e `fim`.
- As listas `x_vals` e `x_names` armazenam, respectivamente, as posições numéricas e os rótulos para a legenda.

In [ ]:
eps = 1e-10   # valor pequeno para evitar a singularidade de tan(β) nos extremos do intervalo

# Calcula os M primeiros autovalores: cada β_m é a raiz de β·tan(β) = H2
# no m-ésimo ramo da tangente, ou seja, no intervalo ((m-1)π, (m-1)π + π/2)
b_m = np.array([
    brentq(lambda b: b * np.tan(b) - H2,
           (m - 1) * np.pi + eps,
           (m - 1) * np.pi + np.pi / 2 - eps)
    for m in range(1, M + 1)
])

A_m = alpha * b_m**2 / L**2                               # A_m = α β_m² / L²
Cm  = 2 * (b_m**2 + H2**2) / (L * (b_m**2 + H2**2 + H2)) # coeficientes de normalização

t_max = 100                           # tempo máximo [s]
t     = np.linspace(0.01, t_max, 800) # 800 instantes de tempo

x_vals  = [0, L/4, L/2, 3*L/4, L]
x_names = ['$x = 0$', '$x = L/4$', '$x = L/2$', '$x = 3L/4$', '$x = L$']

# Exibe os 10 primeiros autovalores para conferência
print('Primeiros 10 autovalores β_m (adimensionais):')
for i in range(10):
    print(f'  β_{i+1:>2} = {b_m[i]:.6f}')

## Definição da função de temperatura

Em Python, uma **função** é definida com a palavra-chave `def`, seguida do nome, dos parâmetros entre parênteses e de dois pontos. O corpo da função deve ser **indentado** (recuado com espaços ou Tab) — a indentação indica ao Python quais linhas pertencem à função. A palavra `return` especifica o valor devolvido.

```python
def nome_da_funcao(parametro):
    resultado = ...     # estas linhas estão dentro da função (indentadas)
    return resultado    # devolve o valor calculado
```

O cálculo da série é vetorizado usando `np.einsum`:
- `np.einsum('m,mt->t', v, M)` multiplica cada linha $m$ da matriz `M` pelo escalar `v[m]` e soma ao longo de $m$, resultando em um vetor de tamanho $N_t$. É equivalente a $\sum_m v_m M_{m,t}$.
- `np.outer(A_m, t)` cria uma matriz $M \times N_t$ onde o elemento $(i,j)$ é $A_i \cdot t_j$.

### Verificação da condição inicial

Para $t \approx 0$, a soma da série deve cancelar exatamente o regime permanente, de modo que $T(x, 0) \approx 0$. Verificamos isso numericamente para três posições.

In [ ]:
def temperatura(x):
    """Retorna T(x, t) para um escalar x e o vetor global t."""
    cos_m  = np.cos(b_m * x / L)                          # cos(β_m x/L), vetor de M valores
    trans  = (alpha * q0 / k) * np.einsum(
        'm,mt->t', Cm * cos_m / A_m,
        np.exp(-np.outer(A_m, t))                          # e^{-A_m t}, matriz M × N_t
    )
    perm   = q0 * (L - x) / k + q0 / h2                   # regime permanente
    return perm - trans

# Verificação da condição inicial T(x, t≈0) ≈ 0
print('Verificação da condição inicial (esperado ≈ 0 °C):')
for x_chk, nome in zip([0.0, L/2, L], ['0', 'L/2', 'L']):
    print(f'  T(x={nome}, t≈0) = {temperatura(x_chk)[0]:.4f} °C')

## Gráfico do campo de temperatura

O laço `for` usa `zip(x_vals, x_names, cores, tracados)` para percorrer as quatro listas ao mesmo tempo.
`zip` funciona como um zíper: emparelha os elementos de mesma posição em cada lista.
Assim, a cada passo do laço, cada variável assume um valor da sua lista correspondente.

A paleta de cores **Okabe-Ito** é acessível a pessoas com daltonismo e é o padrão adotado no livro.

In [ ]:
cores    = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']
tracados = ['-', '--', '-.', ':', (0, (5, 1))]

fig, ax = plt.subplots()   # cria a figura e os eixos

for x, nome, cor, ls in zip(x_vals, x_names, cores, tracados):
    ax.plot(t, temperatura(x), label=nome, color=cor, linestyle=ls, linewidth=1.5)

ax.set_xlabel('Tempo (s)')                      # rótulo do eixo horizontal
ax.set_ylabel('T(x, t)  (°C)')                  # rótulo do eixo vertical
ax.set_title('Problema X23B10T0 — placa com fluxo imposto e superfície convectiva')
ax.legend()                                     # exibe a legenda
ax.grid(True)                                   # grade de fundo
plt.tight_layout()
plt.show()